# AI model comparison — HTML validation report

Compare extracted text artifacts from two PDFs on an Istari Digital system branch with the Anthropic Messages API. The model returns structured JSON; the notebook fills [`report_template.html`](report_template.html) with lineage (artifact ← model), then tracks the HTML report on the branch.

You will:

1. Connect with `istari_labs_helpers` and open a system by **name** and **branch**
2. Name the two **models** and the **artifact** filenames to compare (for example `text.txt` from each PDF)
3. Skip re-running when a prior report exists and neither parent model has a newer revision
4. Resolve each artifact via `find_artifact` — or stop and ask you to run extraction
5. Call Claude, render an HTML report with lineage, track it on the branch, and link the artifacts

Companion to [CAD parameter validation](validation.ipynb).

### Prerequisites

From the cookbook repository root:

```bash
uv sync --group dev --group ai
uv run python -m ipykernel install --user --name istari-client-cookbook-ai --display-name "Python (istari-client-cookbook + ai)"
```

Select the **Python (istari-client-cookbook + ai)** kernel. Reload the window if the kernel picker does not list it yet.

| Group | Packages | Used for |
|---|---|---|
| **`dev`** | `istari-digital-client`, `python-dotenv`, notebook tooling | Connect, systems, models |
| **`ai`** | `anthropic`, `istari-labs-helpers`, `pdfplumber`, `openpyxl`, `python-docx` | Messages API, helpers, document text extraction |

- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`, and `ANTHROPIC_API_KEY` (or `CLAUDE_API_KEY`)
- Two PDF models on the branch with completed extract jobs that produced the named artifacts
- **Branching** enabled when you commit (Istari Digital web app → **Application Settings** → **Experimental Features**)

### Running order

Run cells top to bottom. Later cells no-op when `PROCEED` is false.


## 1 · Connect

Load credentials from [`samples/.env`](../.env). Assert that `ANTHROPIC_API_KEY` is set.


In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display
from istari_labs_helpers import IstariPlatform

NOTEBOOK_DIR = Path.cwd()
_env = NOTEBOOK_DIR / ".env"
if not _env.exists():
    _env = NOTEBOOK_DIR.parent / ".env"
load_dotenv(_env)

assert os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("CLAUDE_API_KEY"), (
    "Set ANTHROPIC_API_KEY (or CLAUDE_API_KEY) in samples/.env"
)
if not os.environ.get("ANTHROPIC_API_KEY") and os.environ.get("CLAUDE_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

platform = IstariPlatform.from_env()
print(f"Connected as {platform.whoami().email}")


## 2 · Prep

Set the system and branch, the two **model** filenames on that branch, and the **artifact** filenames produced by extraction. Set **per-side** extract tool/function (`EXTRACT_TOOL_A` / `EXTRACT_FUNCTION_A`, and the same for B) so helpers can look up completed jobs.


In [ ]:
SYSTEM_NAME = "AI-diff"
BRANCH_NAME = "baseline"

# Models on the branch (filenames) and the extract artifacts to compare.
MODEL_A = "Warthrop_ICD_Rev3.pdf"
ARTIFACT_A = "text.txt"
EXTRACT_TOOL_A = "open_pdf"
EXTRACT_FUNCTION_A = "@istari:extract"

MODEL_B = "Warthrop_SignalDefinitions.xlsx"
ARTIFACT_B = "workbook.html"
EXTRACT_TOOL_B = "open_spreadsheet"
EXTRACT_FUNCTION_B = "@istari:extract"

REPORT_FILENAME = "ai_diff_report.html"
WEBAPP_BASE_URL = "https://demo.istari.app"
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL") or os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-5")

system = platform.get_system(SYSTEM_NAME)
branch = system.get_branch(BRANCH_NAME)


def file_webapp_url(resource_id: str, revision_id: str) -> str:
    # Deep link to a file revision in the Istari Digital web app.
    return f"{WEBAPP_BASE_URL.rstrip('/')}/files/{resource_id}/{revision_id}?tab=file"


print(f"System: {system.name} ({system.id})")
print(f"Branch: {branch.name!r}  snapshot={branch.snapshot_id}")
print(f"Compare: {MODEL_A}/{ARTIFACT_A}  vs  {MODEL_B}/{ARTIFACT_B}")
print(f"Extract A: {EXTRACT_TOOL_A} / {EXTRACT_FUNCTION_A}")
print(f"Extract B: {EXTRACT_TOOL_B} / {EXTRACT_FUNCTION_B}")


## 3 · Models, artifacts, and report freshness

**Concept.** Helpers use the SDK resource type: `is_model` / `is_artifact` (and `as_model()` / `as_artifact()`). Find a named extract product with `model.find_artifact(filename=…)` — that reads the model’s attached artifacts and does **not** call the jobs list API. From an artifact, `.job` recovers the producing job when needed.

This cell:

1. Looks for an existing report on the branch
2. If present, recovers the two artifacts linked last time and checks whether either **parent model** has a newer revision — if not, skips the run
3. Resolves `MODEL_*` / `ARTIFACT_*` via `find_model` + `find_artifact`; stops with a clear prompt if the artifact is missing


In [ ]:
EXISTING_REPORT_RESOURCE_ID = None
PROCEED = False
model_a = model_b = None
art_a = art_b = None
job_a = job_b = None


def _find_report_on_branch():
    for rev in branch.list_revisions():
        name = rev.name or ""
        if name == REPORT_FILENAME and rev.resource_id:
            return rev
    return None


def _linked_source_revision_ids(report_revision_id: str) -> list[str]:
    """Return left-hand revision ids that produce the report (expects two)."""
    v3 = platform.v3
    rel_page = v3.list_revision_relationships(revision_id=report_revision_id, size=50)
    linked = []
    for rel in rel_page.items or []:
        left = rel.left_revision
        right = rel.right_revision
        if (
            left
            and right
            and right.file_revision_id == report_revision_id
            and left.file_revision_id != report_revision_id
        ):
            linked.append(rel)
    linked.sort(key=lambda r: (r.created is not None, r.created))
    return [rel.left_revision.file_revision_id for rel in linked]


def _parent_models_stale(prior_artifact_rev_ids: list[str]) -> bool:
    """True when either artifact's parent model has a newer revision than the job used."""
    print("Last report used:")
    stale = False
    for i, prior_id in enumerate(prior_artifact_rev_ids, start=1):
        resource = platform.get_resource_at_revision(prior_id)
        art = resource.as_artifact() or resource
        job = art.job
        if job is None or not job.model_revision_id:
            print(f"  [{i}] {art.filename or art.name} — no producing job / model revision; treat as stale")
            stale = True
            continue
        parent = platform.get_resource_at_revision(job.model_revision_id)
        is_new = not parent.is_latest
        stale = stale or is_new
        mark = "NEWER model revision" if is_new else "unchanged"
        print(
            f"  [{i}] artifact {art.filename or art.name}  "
            f"← model {parent.filename or parent.name}  "
            f"job_model_rev={job.model_revision_id}  latest={parent.latest_revision.id}  ({mark})"
        )
    return stale


def _resolve_artifact(
    model_filename: str,
    artifact_filename: str,
    extract_tool: str,
    extract_function: str,
):
    """Return (model, job, artifact) or raise with a user-facing message.

    Uses ``find_artifact`` (model.artifacts) — avoids ``list_model_jobs``.
    """
    found = branch.find_model(filename=model_filename)
    if found is None:
        raise RuntimeError(
            f"No model named {model_filename!r} on branch {BRANCH_NAME!r}. "
            "Upload it and track it on the branch first."
        )
    model = found.as_model()
    assert model is not None
    art = model.find_artifact(filename=artifact_filename)
    if art is None:
        names = [
            getattr(a, "name", None)
            for a in (getattr(model.raw, "artifacts", None) or [])
        ]
        raise RuntimeError(
            f"No artifact {artifact_filename!r} on model {model_filename!r}. "
            f"Attached: {names or '(none)'}. Run {extract_tool!r} / {extract_function!r} "
            "in the Istari Digital web app (or via the SDK), then re-run this cell."
        )
    job = art.job  # single get_job; optional for lineage
    return model, job, art


# --- freshness gate when a report already exists ---
report_rev = _find_report_on_branch()
force_run = report_rev is None

if report_rev is not None:
    EXISTING_REPORT_RESOURCE_ID = report_rev.resource_id
    print(
        f"Found report {REPORT_FILENAME!r}  "
        f"revision={report_rev.revision_id}  resource={EXISTING_REPORT_RESOURCE_ID}"
    )
    prior_ids = _linked_source_revision_ids(report_rev.revision_id)
    if len(prior_ids) != 2:
        print(f"Expected 2 linked artifacts on the report; found {len(prior_ids)}. Will resolve by name.")
        force_run = True
    elif not _parent_models_stale(prior_ids):
        print("No new parent-model revisions since the last report — nothing to re-run.")
    else:
        print("Parent model(s) changed — will rebuild the report from latest extracts.")
        force_run = True
else:
    print(f"No report named {REPORT_FILENAME!r} on branch {BRANCH_NAME!r} — will create one.")

# --- resolve models + artifacts by name ---
if force_run:
    try:
        model_a, job_a, art_a = _resolve_artifact(
            MODEL_A, ARTIFACT_A, EXTRACT_TOOL_A, EXTRACT_FUNCTION_A
        )
        model_b, job_b, art_b = _resolve_artifact(
            MODEL_B, ARTIFACT_B, EXTRACT_TOOL_B, EXTRACT_FUNCTION_B
        )
        PROCEED = True
        print("\nResolved pair:")
        for label, model, job, art in (
            ("A", model_a, job_a, art_a),
            ("B", model_b, job_b, art_b),
        ):
            job_part = f"→ job {job.id}  " if job is not None else ""
            print(
                f"  [{label}] model {model.filename} (rev {model.revision_id})  "
                f"{job_part}→ artifact {art.filename} (rev {art.revision_id})"
            )
    except RuntimeError as exc:
        PROCEED = False
        print(f"Stopped: {exc}")


## 4 · Prompts

The **system prompt** sets Istari Digital traceability rules. The **user prompt** is the focus for this run — edit it for your documents.


In [ ]:
SYSTEM_PROMPT = """You are a technical document analyst working inside the Istari Digital Platform.

The documents you are comparing are extracted text artifacts stored in Istari Digital.
Identify each document by its **artifact revision ID** and cite the **parent model** filename when helpful.
Every finding must cite the artifact revision ID it came from so results are fully traceable.

Only use information explicitly stated in the provided documents. Do not infer, assume, or introduce anything not present in the text.
"""

USER_PROMPT = (
    f"Compare artifact A ({ARTIFACT_A} from model {MODEL_A}) "
    f"against artifact B ({ARTIFACT_B} from model {MODEL_B})."
)

if not PROCEED:
    print("Skipped prompts — nothing to compare.")
else:
    print("System prompt:", " ".join(SYSTEM_PROMPT[:90].splitlines()), "…")
    print("User prompt:", USER_PROMPT)


## 5 · Read artifacts and invoke Anthropic

Read the extracted artifact text (already plain text for typical `text.txt` products). Ask Claude for **JSON only** in the matches / conflicts / missing / recommendation schema.


In [ ]:
import json
import tempfile
import textwrap
from datetime import datetime

import anthropic

if not PROCEED:
    print("Skipped Anthropic call — nothing to compare.")
else:
    assert model_a is not None and model_b is not None
    assert art_a is not None and art_b is not None

    def _strip_json_fences(text: str) -> str:
        text = text.strip()
        fenced = re.match(r"^```(?:json)?\s*([\s\S]*?)```\s*$", text, re.IGNORECASE)
        return fenced.group(1).strip() if fenced else text

    # Lineage for prompts + report
    filename_art_a = art_a.filename or ARTIFACT_A
    filename_art_b = art_b.filename or ARTIFACT_B
    filename_model_a = model_a.filename or MODEL_A
    filename_model_b = model_b.filename or MODEL_B
    rev_art_a, rev_art_b = art_a.revision_id, art_b.revision_id
    rev_model_a = (job_a.model_revision_id if job_a else None) or model_a.revision_id
    rev_model_b = (job_b.model_revision_id if job_b else None) or model_b.revision_id
    uuid_art_a, uuid_art_b = art_a.id, art_b.id
    uuid_model_a, uuid_model_b = model_a.id, model_b.id

    text_a = art_a.read_bytes().decode("utf-8", errors="replace")
    text_b = art_b.read_bytes().decode("utf-8", errors="replace")
    print(f"A: {filename_art_a} from {filename_model_a} — {len(text_a)} characters")
    print(f"B: {filename_art_b} from {filename_model_b} — {len(text_b)} characters")

    user_message = textwrap.dedent(
        f"""
        Return JSON only with this schema:
        {{
          "matches": ["..."],
          "conflicts": [{{"item": "...", "value1": "...", "value2": "..."}}],
          "missing": [{{"item": "...", "missing_from": "A|B", "detail": "..."}}],
          "recommendation": "..."
        }}

        Focus:
        {USER_PROMPT}

        --- Document A (artifact rev {rev_art_a}, file {filename_art_a}, model {filename_model_a} rev {rev_model_a}) ---
        {text_a}

        --- Document B (artifact rev {rev_art_b}, file {filename_art_b}, model {filename_model_b} rev {rev_model_b}) ---
        {text_b}
        """
    ).strip()

    client = anthropic.Anthropic()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    message = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=8192,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    raw_reply = "".join(block.text for block in message.content if hasattr(block, "text"))
    diff = json.loads(_strip_json_fences(raw_reply))
    for key in ("matches", "conflicts", "missing", "recommendation"):
        if key not in diff:
            raise KeyError(f"Claude JSON missing key {key!r}")
    print(f"Claude OK — {len(diff['matches'])} matches, {len(diff['conflicts'])} conflicts")


## 6 · Render HTML report

Fill [`report_template.html`](report_template.html) with findings and a **lineage** block: each artifact plus the parent model revision it came from. Write under a temp directory.


In [ ]:
if not PROCEED:
    print("Skipped HTML render — nothing to compare.")
else:
    from string import Template

    href_art_a = file_webapp_url(uuid_art_a, rev_art_a)
    href_art_b = file_webapp_url(uuid_art_b, rev_art_b)
    href_model_a = file_webapp_url(uuid_model_a, rev_model_a)
    href_model_b = file_webapp_url(uuid_model_b, rev_model_b)

    matches_html = "".join(f"<li>{m}</li>" for m in diff["matches"])
    conflicts_html = "".join(
        f'<tr style="border-bottom:1px solid #ddd">'
        f'<td style="padding:8px">{c["item"]}</td>'
        f'<td style="padding:8px">{c["value1"]}</td>'
        f'<td style="padding:8px">{c["value2"]}</td></tr>'
        for c in diff["conflicts"]
    )
    missing_html = "".join(
        f'<li><b>{m["missing_from"]}</b> did not specify {m["item"]}. {m.get("detail", "")}</li>'
        for m in diff["missing"]
    )

    html_report = Template((NOTEBOOK_DIR / "report_template.html").read_text(encoding="utf-8")).substitute(
        artifact_a=filename_art_a,
        artifact_b=filename_art_b,
        model_a=filename_model_a,
        model_b=filename_model_b,
        rev_art_a=rev_art_a,
        rev_art_b=rev_art_b,
        rev_model_a=rev_model_a,
        rev_model_b=rev_model_b,
        href_art_a=href_art_a,
        href_art_b=href_art_b,
        href_model_a=href_model_a,
        href_model_b=href_model_b,
        provider="claude",
        model=ANTHROPIC_MODEL,
        timestamp=timestamp,
        matches_html=matches_html,
        conflicts_html=conflicts_html,
        missing_html=missing_html,
        recommendation=diff["recommendation"],
    )

    REPORT_DIR = Path(tempfile.mkdtemp(prefix="ai-validation-"))
    report_path = REPORT_DIR / REPORT_FILENAME
    report_path.write_text(html_report, encoding="utf-8")

    prompt_audit = REPORT_DIR / (report_path.stem + "_prompt.txt")
    prompt_audit.write_text(
        f"PROMPT\n{'=' * 40}\n{USER_PROMPT}\n\nPROVIDER: claude\nMODEL: {ANTHROPIC_MODEL}\n"
        f"ARTIFACT_A: {filename_art_a} rev={rev_art_a} from MODEL {filename_model_a} rev={rev_model_a}\n"
        f"ARTIFACT_B: {filename_art_b} rev={rev_art_b} from MODEL {filename_model_b} rev={rev_model_b}\n",
        encoding="utf-8",
    )

    print(f"Wrote {report_path.resolve()}")
    print(f"Wrote {prompt_audit.resolve()}")
    display(HTML(html_report))


## 7 · Track the report on the branch

- **First run** — upload a new report model, track it, advance the branch HEAD
- **Later runs** — upload a new revision of the existing report, then advance so the branch picks up LATEST

> Pause in the Istari Digital web app: open the system → confirm the report appears on the branch.


In [ ]:
if not PROCEED:
    print("Skipped report upload — nothing to compare.")
else:
    branch = system.get_branch(BRANCH_NAME)
    version_label = timestamp.replace(" ", "T").replace(":", "")

    if EXISTING_REPORT_RESOURCE_ID:
        updated = platform.client.update_model(
            EXISTING_REPORT_RESOURCE_ID,
            report_path,
            version_name=version_label,
        )
        report_doc = platform.get_model(updated.id)
        branch.advance_to(branch.configuration)
        print(f"Updated report model {report_doc.id} → revision {report_doc.revision_id}")
    else:
        report_doc = platform.upload_model(
            report_path,
            external_id=f"ai-diff-report-{version_label}",
        )
        new_cfg = branch.add_resource(report_doc).save()
        branch.advance_to(new_cfg)
        print(f"Created report model {report_doc.id}  revision={report_doc.revision_id}")
        print(f"Tracked on configuration: {new_cfg.name} ({new_cfg.id})")

    print(f"Advanced branch {BRANCH_NAME!r} → snapshot {branch.snapshot_id}")


## 8 · Link artifacts to the report

Create a `produces` revision relationship from each **artifact** revision to the HTML report. The next run uses these edges to recover lineage and check parent-model freshness.


In [ ]:
if not PROCEED:
    print("Skipped relationships — nothing to compare.")
else:
    assert art_a is not None and art_b is not None
    from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto

    v3 = platform.v3
    rel_types = list(v3.list_revision_relationship_types(size=50).items or [])
    produces = next((t for t in rel_types if t.name == "produces"), None)
    if produces is None:
        raise RuntimeError(
            "No 'produces' relationship type on this registry. "
            f"Available: {[t.name for t in rel_types] or '(none)'}"
        )

    pairs = [
        (art_a.id, art_a.revision_id, f"artifact A ({filename_art_a})"),
        (art_b.id, art_b.revision_id, f"artifact B ({filename_art_b})"),
    ]

    print(f"Relationship type: {produces.name} ({produces.id})")
    print(f"Report resource={report_doc.id}  revision={report_doc.revision_id}\n")

    for resource_id, revision_id, label in pairs:
        rel = v3.create_revision_relationship(
            new_revision_relationship_dto=NewRevisionRelationshipDto(
                relationship_type_id=produces.id,
                left_revision_id=revision_id,
                right_revision_id=report_doc.revision_id,
            )
        )
        print(f"Linked {label}")
        print(f"  left  resource={resource_id}  revision={revision_id}")
        print(f"  right resource={report_doc.id}  revision={report_doc.revision_id}")
        print(f"  relationship id: {getattr(rel, 'id', rel)}")


## Learn more

- [Key Concepts](https://docs.istaridigital.com/intro/key-concepts)
- [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)
- [Anthropic Messages API](https://docs.anthropic.com/en/api/messages)
